# 위치 임베딩: 시퀀스 순서 인코딩
## 절대 위치 vs 상대 위치 임베딩

> **대상**: 트랜스포머를 처음 공부하는 분  
> **목표**: "왜 필요한지"부터 코드로 단계별로 이해하기

---

### 📚 학습 내용
1. 🤔 **왜 위치 임베딩이 필요한가?** — 어텐션의 순열 불변성 문제
2. 📡 **사인파 위치 임베딩 (Sinusoidal)** — 원조 Transformer (2017)
3. 🎓 **학습 가능한 위치 임베딩 (Learned)** — GPT-2, BERT
4. 🔄 **RoPE (Rotary Position Embedding)** — LLaMA, Mistral, Falcon
5. 📏 **ALiBi (Attention with Linear Biases)** — BLOOM, MPT
6. 📊 **방법론 비교 정리**

### 📖 읽는 법
- 각 섹션: **개념 설명 → 단계별 구현 → 시각화/검증** 순서
- 코드의 `# shape: (...)` 주석으로 데이터 흐름을 따라가세요
- 숫자를 외우기보다 **"정보가 어떻게 이동하는가"** 에 집중하세요

---
*Tutorial ID: adv-2-1-1*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 한글 폰트 설정 (운영체제에 맞게 주석 해제)
# ============================================================
# 맥OS:    plt.rcParams['font.family'] = 'AppleGothic'
# 윈도우:  plt.rcParams['font.family'] = 'Malgun Gothic'
# Colab:
#   !apt-get install -y fonts-nanum > /dev/null 2>&1
#   plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.unicode_minus'] = False   # 음수 기호 깨짐 방지

print("라이브러리 로드 완료!")
print(f"NumPy 버전: {np.__version__}")

In [ ]:
# ============================================================
# 0. 왜 위치 임베딩이 필요한가?
# ============================================================
#
# [ 핵심 문제: 어텐션은 순서를 모른다 — 순열 불변성 ]
#
# Self-Attention 공식:
#   Attention(Q, K, V) = softmax( Q @ K^T / sqrt(d_k) ) @ V
#
# 이 수식에 "몇 번째 위치"인지 정보가 전혀 없습니다!
# 토큰 집합 {나는, 밥을, 먹었다} 의 순서가 달라도
# Attention은 같은 결과를 냅니다.

print("=" * 60)
print("문제 시연: Attention은 어순을 구분하지 못한다")
print("=" * 60)

np.random.seed(42)

# ── 3개 토큰, 4차원 임베딩 ───────────────────────────────
# 의미 임베딩 (위치 정보 없음)
tok_na   = np.array([1.0, 0.5, 0.3, 0.8])   # "나는"
tok_bab  = np.array([0.2, 0.9, 0.7, 0.1])   # "밥을"
tok_meok = np.array([0.6, 0.3, 0.9, 0.4])   # "먹었다"

# 시퀀스 A: 나는(0) → 밥을(1) → 먹었다(2)
# 시퀀스 B: 밥을(0) → 나는(1) → 먹었다(2)   (순서만 다름)
seq_A = np.stack([tok_na,  tok_bab, tok_meok])  # shape: (3, 4)
seq_B = np.stack([tok_bab, tok_na,  tok_meok])  # shape: (3, 4)

# 어텐션 가중치 행렬 (임의)
W_Q = np.random.randn(4, 4) * 0.3
W_K = np.random.randn(4, 4) * 0.3

# Q, K 계산 후 Attention Score
d_k = 4
scores_A = (seq_A @ W_Q) @ (seq_A @ W_K).T / np.sqrt(d_k)  # shape: (3,3)
scores_B = (seq_B @ W_Q) @ (seq_B @ W_K).T / np.sqrt(d_k)

print()
print("[시퀀스 A] 나는(0) → 밥을(1) → 먹었다(2)")
print("Attention Score:  scores[i, j] = 위치i가 위치j에 얼마나 주목하는가")
print(np.round(scores_A, 3))

print()
print("[시퀀스 B] 밥을(0) → 나는(1) → 먹었다(2)")
print("Attention Score:")
print(np.round(scores_B, 3))

print()
print("── 검증 ──────────────────────────────────────────────")
# B에서 0번=밥을, A에서 1번=밥을 → 같은 토큰 쌍의 Score가 동일해야 함
print(f"B[0,0] (밥을→밥을) vs A[1,1] (밥을→밥을): "
      f"{scores_B[0,0]:.3f} vs {scores_A[1,1]:.3f}  → 동일! ✓")
print()
print("★ 결론: Attention은 집합으로만 토큰을 처리합니다.")
print("  '나는 밥을 먹었다'와 '밥을 나는 먹었다'를 구분할 수 없습니다!")
print()
print("  해결책 → 위치 임베딩: 각 토큰에 '위치 정보'를 추가합니다.")

---
## 1. 사인파 위치 임베딩 (Sinusoidal Positional Encoding)

2017년 Google 논문 *"Attention is All You Need"* 에서 처음 제안된 방법입니다.

### 해결 방법
각 위치에 고유한 벡터를 만들어 토큰 임베딩에 **더합니다(+)**.

$$PE_{(pos,\; 2i)} \;=\; \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos,\; 2i+1)} \;=\; \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

| 변수 | 의미 |
|------|------|
| `pos` | 토큰의 위치 인덱스 (0, 1, 2, …) |
| `i` | 임베딩 차원 쌍의 인덱스 (0, 1, …, d/2−1) |
| `d_model` | 임베딩 총 차원 수 (예: 64, 512, 768) |

### 💡 시계 비유
- **낮은 차원 (i≈0)**: 초침처럼 빠르게 진동 → 바로 옆 위치 구분
- **중간 차원**: 분침처럼 중간 속도 → 수십 위치 범위 구분
- **높은 차원 (i≈d/2)**: 시침처럼 천천히 진동 → 수백~수천 위치 구분

다양한 주기를 겹쳐 쌓으면 모든 위치가 고유한 벡터를 가집니다.  
(이진수가 자릿값 조합으로 모든 수를 표현하는 것과 같은 원리)

### 장단점
| 장점 | 단점 |
|------|------|
| 파라미터 없음 (수식으로 결정) | 절대 위치만 직접 인코딩 |
| 학습 길이보다 긴 시퀀스도 계산 가능 | 상대 위치는 간접적으로만 표현 |
| 구현이 단순 | — |

In [ ]:
# ============================================================
# 1-A. 주파수(주기) 개념 먼저 이해하기
# ============================================================
#
# 수식의 분모 10000^(2i/d_model) 이 클수록 주기가 길어집니다.
#
# 이진법 비유:
#   1의 자리:  0 1 0 1 0 1 ...  주기=2  (가장 빠름)
#   2의 자리:  0 0 1 1 0 0 ...  주기=4
#   4의 자리:  0 0 0 0 1 1 ...  주기=8
#   → 다양한 주기의 비트 조합으로 모든 수를 고유하게 표현!
#
# 사인파 임베딩도 같은 원리로 다양한 주기의 파동을 조합합니다.

print("차원(i)별 분모 크기와 주기(period):")
print("-" * 60)
print(f"  {'i':>3}  {'분모 10000^(2i/d)':>22}  {'sin 파동 주기':>14}")
print("-" * 60)

d_demo = 8   # 설명용 (실제는 64, 512 등)

for i in range(d_demo // 2):
    denominator = 10000 ** (2 * i / d_demo)
    period      = 2 * np.pi * denominator    # 한 주기를 완성하는 위치 수
    bar = '-' * min(int(period / 4), 28)
    print(f"  i={i}:  분모={denominator:>10.1f}  주기={period:>10.1f}  |{bar}")

print()
print("핵심 관찰:")
print("  i=0: 분모=1,    주기≈6     → 초침 (세밀한 위치 구분)")
print("  i=1: 분모≈6,    주기≈40")
print("  i=3: 분모≈1000, 주기≈6283  → 시침 (넓은 범위 구분)")
print()
print("이 다양한 주기들의 조합으로")
print("각 위치마다 완전히 다른 고유한 벡터가 만들어집니다.")

In [ ]:
# ============================================================
# 1-B. 사인파 위치 임베딩 구현 (단계별 shape 추적)
# ============================================================

def sinusoidal_encoding(max_len, d_model):
    """
    사인파 위치 임베딩 행렬을 생성합니다.

    Parameters
    ----------
    max_len  : int  최대 시퀀스 길이 (예: 128이면 토큰 128개까지 지원)
    d_model  : int  임베딩 차원 수   (짝수여야 함)

    Returns
    -------
    pe : ndarray, shape (max_len, d_model)
         pe[pos, dim] = 위치 pos의 dim번째 차원 값
    """

    # ── Step 1: 0으로 초기화 ─────────────────────────────────────
    # 행(row) = 위치 인덱스 (0 ~ max_len-1)
    # 열(col) = 임베딩 차원 (0 ~ d_model-1)
    pe = np.zeros((max_len, d_model))
    # pe.shape: (max_len, d_model)

    # ── Step 2: 위치 인덱스 열벡터 만들기 ────────────────────────
    # [:, np.newaxis] → (max_len,) 를 (max_len, 1) 로 변환
    # 나중에 div_term shape (d_model//2,) 와 브로드캐스팅하기 위해
    position = np.arange(max_len)[:, np.newaxis]
    # position.shape: (max_len, 1)

    # ── Step 3: 나눗셈 항(div_term) 계산 ─────────────────────────
    # div_term[i] = 1 / 10000^(2i/d_model)
    # 수치 안정성을 위해 지수-로그 변환:
    #   1/10000^(2i/d) = exp(-(2i/d) * log(10000))
    dim_indices = np.arange(0, d_model, 2)   # [0, 2, 4, ..., d_model-2] = 2i
    div_term    = np.exp(dim_indices * (-np.log(10000.0) / d_model))
    # div_term.shape: (d_model//2,)
    # div_term[0] ≈ 1.0   (i=0, 가장 빠른 진동)
    # div_term[-1] ≈ 0.0001 (i=d/2-1, 가장 느린 진동)

    # ── Step 4: 짝수 차원=sin, 홀수 차원=cos ─────────────────────
    # 브로드캐스팅: (max_len,1) * (d_model//2,) → (max_len, d_model//2)
    pe[:, 0::2] = np.sin(position * div_term)  # 짝수 열: 0,2,4,...
    pe[:, 1::2] = np.cos(position * div_term)  # 홀수 열: 1,3,5,...
    # pe.shape: (max_len, d_model)  — 완성!

    return pe

# ─── 실행 및 확인 ─────────────────────────────────────────────────
print("=" * 60)
print("사인파 위치 임베딩 생성 — shape 추적")
print("=" * 60)

max_len = 128
d_model = 64

pe_tmp = np.zeros((max_len, d_model))
pos_tmp = np.arange(max_len)[:, np.newaxis]
dim_tmp = np.arange(0, d_model, 2)
div_tmp = np.exp(dim_tmp * (-np.log(10000.0) / d_model))

print(f"Step 1 | pe 초기화    shape: {pe_tmp.shape}")
print(f"Step 2 | position     shape: {pos_tmp.shape}  값: {pos_tmp[:5,0]}")
print(f"Step 3 | div_term     shape: {div_tmp.shape}")
print(f"         처음 4개:  {np.round(div_tmp[:4], 4)}")
print(f"         마지막 4개: {np.round(div_tmp[-4:], 6)}")
print(f"Step 4 | sin/cos 채우기 → pe  shape: ({max_len}, {d_model})")

# 실제 생성
pe = sinusoidal_encoding(max_len, d_model)

print()
print("위치별 임베딩 (처음 8차원만 표시):")
for pos in [0, 1, 2, 5, 10, 50]:
    vals = np.round(pe[pos, :8], 3)
    print(f"  pos {pos:3d}: {vals}")
print()
print(f"값의 범위: [{pe.min():.4f}, {pe.max():.4f}]  (sin/cos → 항상 [-1, +1])")

In [ ]:
# ============================================================
# 1-C. 사인파 위치 임베딩 시각화 (4가지 그래프)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Sinusoidal Positional Encoding — Visualization", fontsize=14, fontweight='bold')

# ── 그래프 1: PE 행렬 전체 히트맵 ─────────────────────────────────
ax = axes[0, 0]
im = ax.imshow(pe[:64, :], aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xlabel("Embedding Dimension (d)")
ax.set_ylabel("Token Position (pos)")
ax.set_title("PE Matrix Heatmap\n"
             "row=position, col=dim, color=value(-1~+1)")
plt.colorbar(im, ax=ax, label='value', shrink=0.9)
for x_line in [0, 16, 32, 48, 64]:
    ax.axvline(x=x_line - 0.5, color='white', linewidth=0.5, linestyle='--', alpha=0.4)
ax.text(8,  67, "Low dim\n(fast)", ha='center', fontsize=8, color='gray')
ax.text(56, 67, "High dim\n(slow)", ha='center', fontsize=8, color='gray')

# ── 그래프 2: 차원별 진동 패턴 ────────────────────────────────────
ax = axes[0, 1]
pos_range = np.arange(64)
examples = [
    (0,  'crimson',     'dim  0  (i=0, fastest)'),
    (2,  'darkorange',  'dim  2  (i=1)'),
    (16, 'forestgreen', 'dim 16  (i=8, middle)'),
    (62, 'steelblue',   'dim 62  (i=31, slowest)'),
]
for dim, color, label in examples:
    ax.plot(pos_range, pe[:64, dim], color=color, label=label, linewidth=1.8)
ax.set_xlabel("Token Position")
ax.set_ylabel("PE Value")
ax.set_title("Oscillation per Dimension\n"
             "Low dim=fast freq, High dim=slow freq")
ax.legend(fontsize=8)
ax.axhline(y=0, color='black', linewidth=0.4, alpha=0.5)
ax.grid(alpha=0.2)

# ── 그래프 3: 기준 위치와의 코사인 유사도 ──────────────────────────
ax = axes[1, 0]
ref_pos = 10

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

sims = [cosine_sim(pe[ref_pos], pe[p]) for p in range(max_len)]
ax.plot(sims, color='steelblue', linewidth=1.5)
ax.axvline(x=ref_pos, color='red', linewidth=2, linestyle='--',
           label=f'Reference pos={ref_pos}')
ax.fill_between(range(max_len), sims, alpha=0.15, color='steelblue')
ax.set_xlabel("Token Position")
ax.set_ylabel("Cosine Similarity")
ax.set_title(f"Similarity from pos={ref_pos} to all positions\n"
             "Closer positions = higher similarity")
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# ── 그래프 4: 토큰 임베딩 + 위치 임베딩 합산 ─────────────────────
ax = axes[1, 1]
np.random.seed(7)
d_vis = 8
tok_v = np.random.randn(d_vis) * 0.5  # 예시 토큰 임베딩
pos_v = pe[3, :d_vis]                  # 위치 3의 임베딩
fin_v = tok_v + pos_v                  # 덧셈으로 결합

x_idx = np.arange(d_vis)
w = 0.25
ax.bar(x_idx - w, tok_v, width=w, label='Token Emb.',    color='#3498db', alpha=0.8)
ax.bar(x_idx,     pos_v, width=w, label='Position Emb.', color='#e74c3c', alpha=0.8)
ax.bar(x_idx + w, fin_v, width=w, label='Final Input',   color='#27ae60', alpha=0.8)
ax.set_xlabel("Dimension Index")
ax.set_ylabel("Value")
ax.set_title("How PE is Applied (pos=3)\n"
             "Final Input = Token Emb. + Position Emb.")
ax.legend(fontsize=9)
ax.axhline(y=0, color='black', linewidth=0.4)
ax.set_xticks(x_idx)
ax.grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 1-D. 사인파 임베딩의 3가지 핵심 속성 분석
# ============================================================

def cosine_sim(a, b):
    """두 벡터의 코사인 유사도. 범위: [-1, +1]"""
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return np.dot(a, b) / (norm + 1e-8)

# ── 속성 1: 가까운 위치 = 유사한 벡터 ────────────────────────────
print("=" * 60)
print("속성 1: 가까운 위치일수록 코사인 유사도가 높다")
print("  → 모델이 '두 토큰이 얼마나 가까운가'를 벡터 유사도로 파악 가능")
print("=" * 60)
print(f"  {'기준':^5}  {'비교':^5}  {'코사인 유사도':^14}  시각적 표현")
print("-" * 60)

ref = 0
for pos in [1, 2, 3, 5, 10, 20, 50, 100, 127]:
    sim    = cosine_sim(pe[ref], pe[pos])
    filled = max(0, int(sim * 20))
    bar    = "█" * filled + "░" * (20 - filled)
    print(f"  pos {ref:3d} vs pos {pos:3d}:   {sim:+.4f}   |{bar}|")

# ── 속성 2: 외삽 가능 ─────────────────────────────────────────────
print()
print("=" * 60)
print("속성 2: 학습 길이를 초과해도 계산 가능 (외삽)")
print("  → 수식 기반이므로 어떤 위치든 공식 적용 가능")
print("=" * 60)
ext_pe = sinusoidal_encoding(300, d_model)
print(f"  원래 max_len={max_len}  이지만 더 긴 위치도 계산:")
for pos in [128, 200, 299]:
    vals = np.round(ext_pe[pos, :6], 3)
    print(f"  pos {pos:3d}: {vals}  ✓")

# ── 속성 3: 삼각함수 덧셈 공식으로 상대 위치 표현 ─────────────────
print()
print("=" * 60)
print("속성 3: 상대 위치가 삼각함수 내적으로 표현됨 (수학적 검증)")
print("  공식: sin(a)*sin(b) + cos(a)*cos(b) = cos(a - b)")
print("  → pe[a,0]*pe[b,0] + pe[a,1]*pe[b,1] = cos(a-b) 검증")
print("=" * 60)
for pos_a, pos_b in [(0,3), (5,8), (10,3), (20,25), (0,0)]:
    lhs = pe[pos_a, 0]*pe[pos_b, 0] + pe[pos_a, 1]*pe[pos_b, 1]
    rhs = np.cos(pos_a - pos_b)
    ok  = "✓" if abs(lhs - rhs) < 1e-6 else "✗"
    print(f"  a={pos_a:2d}, b={pos_b:2d} | LHS={lhs:+.5f}, cos({pos_a:2d}-{pos_b:2d})={rhs:+.5f} {ok}")

print()
print("→ 상대 거리 (a-b) 가 같으면 내적값도 같습니다!")
print("  이 덕분에 어텐션이 상대 위치를 (간접적으로) 학습할 수 있습니다.")

---
## 2. 학습 가능한 위치 임베딩 (Learned Positional Embeddings)

GPT-2, BERT 등이 사용하는 방법으로, 위치 임베딩을 **학습으로 결정**합니다.

### 아이디어
- 토큰 임베딩 테이블(단어 → 벡터)처럼, **위치 임베딩 테이블(위치 ID → 벡터)** 을 만들어 학습
- 위치 0, 1, 2, ...에 대응하는 벡터를 역전파(backpropagation)로 최적화

```
position_ids = [0, 1, 2, 3, 4]           # 5개 토큰의 위치
pos_embed    = table[position_ids]         # 테이블에서 해당 벡터 조회
final_input  = token_embed + pos_embed    # 덧셈으로 결합 → Transformer 입력
```

### 장단점
| 장점 | 단점 |
|------|------|
| 데이터에 최적화된 표현 학습 | 학습 때 본 최대 길이 이상은 **불가 (외삽 X)** |
| 구현이 매우 단순 (Lookup Table) | 파라미터 수 증가 (max_len × d_model) |
| 특정 태스크에서 사인파보다 좋을 수 있음 | 긴 문서 처리에 취약 |

### 사용 모델
- **GPT-2**: 1024 위치 × 768차원 = **786,432개** 파라미터 (위치 임베딩만)
- **BERT**: 512 위치 × 768차원 = **393,216개** 파라미터
- 실제 BERT 논문에서 사인파와 학습형의 **최종 성능이 거의 동등**했다고 보고

In [ ]:
# ============================================================
# 2. 학습 가능한 위치 임베딩 구현
# ============================================================

print("=" * 60)
print("2. 학습 가능한 위치 임베딩 (Learned Positional Embeddings)")
print("=" * 60)

np.random.seed(0)

max_pos = 10   # 이 모델이 지원하는 최대 토큰 수 (예시용으로 작게)
d_emb   = 8    # 임베딩 차원

# ─────────────────────────────────────────────────────────────────
# 위치 임베딩 테이블 초기화
#   - shape: (max_pos, d_emb)
#   - 실제 모델에서는 역전파로 계속 업데이트되는 학습 파라미터
#   - 초기화: 표준편차 0.02 (BERT, GPT-2의 일반적 초기화 방식)
# -----------------------------------------------------------------
position_table = np.random.randn(max_pos, d_emb) * 0.02
# position_table[i] = 위치 i에 해당하는 d_emb차원 벡터

print(f"\n위치 임베딩 테이블: shape={position_table.shape}")
print(f"  → {max_pos}개 위치 각각 {d_emb}차원 벡터 보유")
print()
print("학습 전 테이블 (아직 의미 없는 랜덤 초기화):")
for i in [0, 1, 2]:
    print(f"  pos {i}: {np.round(position_table[i], 4)}")
print("  ...")

# ── 실제 사용: 5개 토큰 시퀀스 처리 ──────────────────────────────
print()
print("[ 5개 토큰 처리 흐름 ]")
print()
seq_len = 5
np.random.seed(42)

# ① 토큰 임베딩 (단어의 '의미' 벡터)
token_embed = np.random.randn(seq_len, d_emb)
print(f"① 토큰 임베딩       shape: {token_embed.shape}  ← 단어 의미")

# ② 위치 ID 생성 및 테이블 조회
position_ids = np.arange(seq_len)                # [0, 1, 2, 3, 4]
pos_embed    = position_table[position_ids]       # 조회(Lookup)
# shape: (5, 8)
print(f"② 위치 ID 생성      : {position_ids}")
print(f"   위치 임베딩 조회  shape: {pos_embed.shape}  ← 위치 정보")

# ③ 최종 입력 = 토큰 임베딩 + 위치 임베딩 (element-wise 덧셈)
final_input = token_embed + pos_embed             # shape: (5, 8)
print(f"③ 최종 입력 (합산)  shape: {final_input.shape}  ← Transformer에 입력")

# ── 학습 전 vs 사인파 유사도 비교 ────────────────────────────────
print()
print("[ 학습 전 vs 사인파: 위치 간 코사인 유사도 비교 ]")
sin_pe_demo = sinusoidal_encoding(max_pos, d_emb)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

pairs = [(0,1, "pos0 vs pos1"), (0,5, "pos0 vs pos5"), (0,9, "pos0 vs pos9")]
print(f"  {'쌍':<18} {'사인파':>10} {'학습형(학습전)':>14}")
print("  " + "-" * 44)
for i, j, label in pairs:
    s_sim = cosine_sim(sin_pe_demo[i], sin_pe_demo[j])
    l_sim = cosine_sim(position_table[i], position_table[j])
    print(f"  {label:<18} {s_sim:>+10.4f} {l_sim:>+14.4f}")

print()
print("  → 사인파는 처음부터 '가까운 위치=유사'한 구조를 가집니다.")
print("    학습형은 학습 후 유사한 구조를 습득합니다.")

# ── 단점 시연 ─────────────────────────────────────────────────────
print()
print("[ 치명적 단점: 범위 초과 시 오류 ]")
print(f"  이 모델의 최대 길이: {max_pos}개 토큰")
try:
    _ = position_table[max_pos]
except IndexError as e:
    print(f"  position_table[{max_pos}] → IndexError 발생!")
    print(f"  → 학습 때 본 최대 길이({max_pos-1}번 위치)를 넘으면 처리 불가!")
    print(f"    (사인파는 이런 제한이 없습니다)")

---
## 3. RoPE (Rotary Position Embedding)

LLaMA, Mistral, Falcon, GPT-NeoX 등 최신 LLM 대부분이 채택한 방법입니다.

### 기존 방법과의 차이
| 방법 | 위치 정보 적용 방식 |
|------|-------------------|
| Sinusoidal / Learned | 토큰 임베딩에 위치 벡터를 **더함 (+)** |
| **RoPE** | Query/Key 벡터를 위치에 따라 **회전** |

### 2D 회전 행렬 복습
벡터 $(x, y)$를 각도 $\theta$만큼 회전:

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix}$$

- $x' = x\cos\theta - y\sin\theta$
- $y' = x\sin\theta + y\cos\theta$
- **크기(norm)는 항상 유지**: 방향만 바뀜

### 핵심 수학적 보장
위치 $m$의 Query $q_m$ 과 위치 $n$의 Key $k_n$ 의 내적:

$$q_m^T k_n = f(q,\; k,\; m-n)$$

**오직 상대 거리 $(m-n)$에만 의존!** 절대 위치 $m$, $n$ 자체는 무관합니다.

### RoPE 동작 방식
1. $d$차원 벡터를 $d/2$쌍의 2D 벡터로 분리: $(x_0,x_1),\,(x_2,x_3),\,\ldots$
2. 각 쌍 $i$를 각도 $\theta_i = pos / 10000^{2i/d}$ 로 회전
3. 다양한 주파수 사용 (사인파 임베딩과 동일한 멀티스케일 철학)

In [ ]:
# ============================================================
# 3-A. 2D 회전 개념 이해 (RoPE의 수학적 기초)
# ============================================================
#
# RoPE의 핵심은 "두 벡터를 서로 다른 각도로 회전시키면
# 내적이 그 각도의 차이에만 의존한다"는 성질입니다.

print("[ 2D 회전 기본 동작 ]")
print("-" * 50)

def rotate_2d(x, y, theta_rad):
    """
    2D 벡터 (x, y)를 theta_rad 라디안만큼 반시계 방향 회전.
    회전 행렬: [[cos, -sin], [sin, cos]]
    """
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return x * c - y * s, x * s + y * c

x0, y0 = 1.0, 0.0
print(f"원래 벡터: ({x0:.3f}, {y0:.3f}),  크기={np.sqrt(x0**2+y0**2):.3f}")
print()
for deg in [0, 30, 45, 90, 180, 270]:
    rad  = np.deg2rad(deg)
    xr, yr = rotate_2d(x0, y0, rad)
    norm = np.sqrt(xr**2 + yr**2)
    print(f"  {deg:3d}도 회전 → ({xr:+.3f}, {yr:+.3f}),  크기={norm:.3f}")
print()
print("→ 방향은 바뀌지만 크기는 항상 1.0 유지 (회전은 등거리 변환)")

# ── 핵심 성질: 같은 각도 회전 → 내적 보존 ────────────────────────
print()
print("[ 핵심 성질: 같은 각도로 회전해도 내적은 그대로 ]")
print("수학: (R·a)^T (R·b) = a^T R^T R b = a^T b  (R^T R = I, 직교행렬)")
print("-" * 50)

a = np.array([1.0, 2.0])
b = np.array([3.0, 1.0])
orig = np.dot(a, b)
print(f"a={a},  b={b}")
print(f"원래 내적 a·b = {orig:.4f}")
print()
for deg in [0, 30, 90, 180]:
    rad  = np.deg2rad(deg)
    ar   = np.array(rotate_2d(a[0], a[1], rad))
    br   = np.array(rotate_2d(b[0], b[1], rad))
    dot  = np.dot(ar, br)
    same = "✓ 동일" if abs(dot - orig) < 1e-6 else "✗ 다름"
    print(f"  {deg:3d}도 같은 각도 회전 후 내적: {dot:.4f}  {same}")

print()
print("[ 다른 각도로 회전하면 내적이 달라진다 ]")
print(f"원래 a·b = {orig:.4f}")
for deg_a, deg_b in [(0,30), (0,90), (30,90), (45,135)]:
    ar = np.array(rotate_2d(a[0], a[1], np.deg2rad(deg_a)))
    br = np.array(rotate_2d(b[0], b[1], np.deg2rad(deg_b)))
    dot = np.dot(ar, br)
    diff = deg_a - deg_b
    print(f"  a: {deg_a:3d}도, b: {deg_b:3d}도 → 내적: {dot:.4f}  (각도차={diff:+d}도)")

print()
print("★ RoPE 핵심:")
print("  Query를 '위치 m의 각도'로, Key를 '위치 n의 각도'로 회전")
print("  → q·k 내적이 '상대 위치 (m-n)'에만 의존하게 됩니다!")

In [ ]:
# ============================================================
# 3-B. RoPE 구현 (단계별 상세 주석)
# ============================================================

def apply_rope(x, pos, base=10000):
    """
    벡터 x를 위치 pos에 해당하는 회전으로 변환합니다. (RoPE 적용)

    Parameters
    ----------
    x    : ndarray shape (d,)  Query 또는 Key 벡터 (d는 짝수)
    pos  : int                 이 벡터의 절대 위치 인덱스
    base : int                 주파수 기저 (기본값 10000)

    Returns
    -------
    x_rot : ndarray shape (d,)  회전이 적용된 벡터

    동작:
      d차원 벡터를 d/2 쌍으로 분리: (x0,x1), (x2,x3), ...
      쌍 i → theta_i = pos / base^(2i/d) 각도로 2D 회전
    """
    d = len(x)   # 차원 수 (짝수여야 함)

    # ── Step 1: 각 차원 쌍(i)의 회전 각도 계산 ──────────────────
    # theta_i = pos / base^(2i/d)
    # i=0: theta=pos/1      → 빠른 회전 (위치마다 많이 돌아감)
    # i가 클수록: theta 작아짐 → 느린 회전 (위치 달라도 조금만 변화)
    i_idx  = np.arange(d // 2)                    # [0, 1, ..., d//2-1]
    theta  = pos / (base ** (2 * i_idx / d))      # shape: (d//2,)
    cos_t  = np.cos(theta)                         # shape: (d//2,)
    sin_t  = np.sin(theta)                         # shape: (d//2,)

    # ── Step 2: 각 쌍(x_{2i}, x_{2i+1})에 2D 회전 적용 ──────────
    x_rot = np.zeros_like(x)
    for i in range(d // 2):
        xe = x[2 * i]        # 짝수 인덱스 원소 ("x 성분")
        xo = x[2 * i + 1]   # 홀수 인덱스 원소 ("y 성분")
        # 회전 공식: x' = x·cos - y·sin,  y' = x·sin + y·cos
        x_rot[2 * i]     = xe * cos_t[i] - xo * sin_t[i]
        x_rot[2 * i + 1] = xe * sin_t[i] + xo * cos_t[i]

    return x_rot

# ── 기본 동작 확인 ─────────────────────────────────────────────
print("=" * 60)
print("RoPE 기본 동작 — 위치별 내적값 변화")
print("=" * 60)

np.random.seed(42)
d_head = 8
q = np.random.randn(d_head)
k = np.random.randn(d_head)

print(f"Query: {np.round(q, 3)}")
print(f"Key:   {np.round(k, 3)}")
print(f"원래 내적 q·k: {np.dot(q, k):.4f}")
print()

print(f"{'pos_q':>7}  {'pos_k':>7}  {'내적(회전후)':>13}  {'gap':>5}  비고")
print("-" * 58)
for pos_q, pos_k in [(0,0),(5,5),(10,10),(0,5),(5,10),(0,10),(3,3)]:
    qr  = apply_rope(q, pos_q)
    kr  = apply_rope(k, pos_k)
    dot = np.dot(qr, kr)
    gap = pos_q - pos_k
    note = "gap=0 (같은 위치)" if gap == 0 else f"gap={gap:+d}"
    print(f"  q(pos={pos_q:2d}) · k(pos={pos_k:2d}):  {dot:+.4f}   {note}")

print()
print("관찰:")
print("  - gap=0인 (0,0), (5,5), (10,10): 내적값이 동일합니다.")
print("  - gap=-5인 (0,5), (5,10): 내적값이 동일합니다.")
print("  → 절대 위치가 아닌 '상대 거리(gap)'가 내적을 결정합니다!")

In [ ]:
# ============================================================
# 3-C. RoPE 핵심 속성 수치 검증
# ============================================================

print("=" * 60)
print("검증: gap(상대 거리)이 같으면 내적이 항상 같다")
print("=" * 60)

np.random.seed(7)
d_head = 16   # 좀 더 현실적인 차원
q = np.random.randn(d_head)
k = np.random.randn(d_head)

# ── 검증 1: gap=5인 여러 절대 위치 쌍 ─────────────────────────
print()
print("[ gap=+5 인 경우: pos_q - pos_k = 5 ]")
print("  절대 위치는 다르지만 상대 거리가 같으면 내적이 같아야 함")
print()
dots_gap5 = []
for pos_q, pos_k in [(5,0),(10,5),(20,15),(50,45),(100,95)]:
    qr  = apply_rope(q, pos_q)
    kr  = apply_rope(k, pos_k)
    dot = np.dot(qr, kr)
    dots_gap5.append(dot)
    print(f"  q(pos={pos_q:3d}) · k(pos={pos_k:3d}): {dot:+.8f}")
print(f"  표준편차(std): {np.std(dots_gap5):.2e}  ← 거의 0 → 모두 동일! ✓")

# ── 검증 2: gap=-3인 여러 쌍 ─────────────────────────────────
print()
print("[ gap=-3 인 경우: pos_q - pos_k = -3 ]")
dots_neg3 = []
for pos_q, pos_k in [(0,3),(5,8),(10,13),(20,23)]:
    qr  = apply_rope(q, pos_q)
    kr  = apply_rope(k, pos_k)
    dot = np.dot(qr, kr)
    dots_neg3.append(dot)
    print(f"  q(pos={pos_q:2d}) · k(pos={pos_k:2d}): {dot:+.8f}")
print(f"  표준편차(std): {np.std(dots_neg3):.2e}  ← 거의 0 → 동일! ✓")

# ── 검증 3: gap이 다르면 내적이 달라짐 ───────────────────────
print()
print("[ gap이 달라지면 내적값도 달라진다 ]")
print("  (q는 pos=10 고정, k의 위치만 변경)")
qr10 = apply_rope(q, 10)
for pos_k in [10, 7, 5, 2, 0, 15, 20]:
    kr  = apply_rope(k, pos_k)
    dot = np.dot(qr10, kr)
    gap = 10 - pos_k
    print(f"  q(pos=10) · k(pos={pos_k:2d}): {dot:+.6f}   gap={gap:+d}")

# ── 크기 보존 확인 ────────────────────────────────────────────
print()
print("[ 추가: RoPE는 벡터 크기(L2 norm)를 보존합니다 ]")
orig_norm = np.linalg.norm(q)
for pos in [0, 5, 100, 1000]:
    rot_norm = np.linalg.norm(apply_rope(q, pos))
    print(f"  pos={pos:4d}: 원래={orig_norm:.6f}, 회전후={rot_norm:.6f}  ✓")

In [ ]:
# ============================================================
# 3-D. RoPE 시각화
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("RoPE (Rotary Position Embedding) — Visualization",
             fontsize=13, fontweight='bold')

np.random.seed(42)
d_head = 8
q = np.random.randn(d_head)
k = np.random.randn(d_head)

# ── 그래프 1: Key 위치별 내적값 변화 ─────────────────────────────
ax = axes[0]
q_rot_ref = apply_rope(q, 10)   # q는 pos=10 고정
positions_k = list(range(25))
dots = [np.dot(q_rot_ref, apply_rope(k, p)) for p in positions_k]

bar_colors = ['#27ae60' if p == 10 else '#3498db' for p in positions_k]
ax.bar(positions_k, dots, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(x=10, color='red', linestyle='--', linewidth=2, label='q pos=10')
ax.set_xlabel("Key Position (pos_k)")
ax.set_ylabel("Dot Product q·k")
ax.set_title("q(pos=10) dot k(pos=n)\n"
             "Green=same position, Blue=others")
ax.legend(fontsize=9)
ax.grid(alpha=0.25, axis='y')

# ── 그래프 2: 2D 공간에서의 회전 궤적 ───────────────────────────
ax = axes[1]
q2d    = q[:2]
norm2d = np.linalg.norm(q2d)

# 원(회전 궤적)
theta_c = np.linspace(0, 2 * np.pi, 200)
ax.plot(norm2d * np.cos(theta_c), norm2d * np.sin(theta_c),
        color='lightgray', linewidth=1, linestyle='--', label='rotation path')

# 위치별 회전 벡터
pos_vis  = [0, 3, 6, 10, 15, 20]
cmap_vis = plt.cm.plasma(np.linspace(0.1, 0.9, len(pos_vis)))
for pos, color in zip(pos_vis, cmap_vis):
    # 2D로 rotate (전체 벡터를 만들어서 처음 2차원만 사용)
    q_full    = np.zeros(d_head)
    q_full[:2] = q2d
    q_rot2d   = apply_rope(q_full, pos)[:2]
    ax.annotate("", xy=q_rot2d, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0))
    ax.text(q_rot2d[0]*1.25, q_rot2d[1]*1.25, f'p={pos}',
            fontsize=8, color=color, ha='center')

ax.set_aspect('equal')
ax.axhline(y=0, color='black', linewidth=0.4)
ax.axvline(x=0, color='black', linewidth=0.4)
ax.set_title("Query Vector Rotation (dim 0-1)\n"
             "Each position = different angle")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# ── 그래프 3: 상대 거리 vs 내적값 ───────────────────────────────
ax = axes[2]
rel_dists = list(range(-12, 13))
q_center  = apply_rope(q, 12)
dots_rel  = [np.dot(q_center, apply_rope(k, 12 + d)) for d in rel_dists]

bar_colors2 = ['#e74c3c' if d < 0 else '#2ecc71' if d == 0 else '#3498db'
               for d in rel_dists]
ax.bar(rel_dists, dots_rel, color=bar_colors2, alpha=0.8, edgecolor='white', linewidth=0.3)
ax.axvline(x=0, color='black', linewidth=1.5)
ax.set_xlabel("Relative Distance (pos_q - pos_k)")
ax.set_ylabel("Dot Product")
ax.set_title("Inner product by relative distance\n"
             "Red=past, Green=self, Blue=future")
ax.grid(alpha=0.25, axis='y')

plt.tight_layout()
plt.show()

print("그래프 3 핵심 해석:")
print("  • 같은 상대 거리(같은 x 위치)라면 절대 위치에 상관없이 내적이 동일")
print("  • RoPE 덕분에 Attention이 자동으로 '상대적 거리'를 학습")

---
## 4. ALiBi (Attention with Linear Biases)

BLOOM, MPT 등이 채택한 방법으로, **위치 임베딩 자체를 없애고** Attention Score에 직접 편향을 추가합니다.

### 핵심 아이디어
위치 벡터를 따로 만드는 대신, Attention Score 계산 시 **거리에 비례하는 음수 패널티**를 더합니다.

**기존 Attention:**
$$\text{Score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}}$$

**ALiBi 적용 후:**
$$\text{Score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} \;-\; m \cdot |i - j|$$

| 기호 | 의미 |
|------|------|
| $m$ | 헤드별 고정 기울기(slope), 학습 안 함 |
| $\|i-j\|$ | Query 위치 $i$ 와 Key 위치 $j$ 사이의 거리 |
| $m$ 클수록 | 패널티 강함 → 가까운 토큰 위주로 봄 |
| $m$ 작을수록 | 패널티 약함 → 멀리 있는 토큰도 봄 |

### 헤드별 기울기 공식
헤드 $h$ (1-indexed), 총 $H$개 헤드:
$$m_h = 2^{-8h/H}$$

→ 하나의 Transformer 레이어에서 헤드들이 **서로 다른 시야 거리**를 담당

### 장점
- 구현이 가장 단순 (기존 코드에 bias 행렬 하나만 추가)
- 학습 파라미터 없음
- 긴 시퀀스 외삽 성능이 특히 우수

In [ ]:
# ============================================================
# 4-A. ALiBi 구현 (단계별 상세 주석)
# ============================================================

def alibi_bias(seq_len, num_heads):
    """
    ALiBi 편향 행렬을 생성합니다.

    Parameters
    ----------
    seq_len   : int  시퀀스 길이
    num_heads : int  어텐션 헤드 수

    Returns
    -------
    biases : ndarray shape (num_heads, seq_len, seq_len)
             biases[h, i, j] = 헤드 h에서 위치 i → 위치 j 볼 때의 편향
    """

    # ── Step 1: 헤드별 기울기(slope) 계산 ──────────────────────
    # m_h = 2^(-8h/H),  h = 1, 2, ..., num_heads
    # h가 작을수록 m 큼 → 강한 패널티 → 가까운 토큰 위주
    # h가 클수록  m 작음 → 약한 패널티 → 멀리도 봄
    h_idx  = np.arange(1, num_heads + 1)             # [1, 2, ..., H]
    slopes = 2.0 ** (-8.0 * h_idx / num_heads)        # shape: (num_heads,)

    # ── Step 2: 위치 간 거리 행렬 ───────────────────────────────
    # distances[i, j] = i - j
    #   i > j (양수): j는 i보다 앞 (과거 토큰) → 현재 허용
    #   i = j (0):    자기 자신
    #   i < j (음수): j는 i보다 뒤 (미래 토큰) → 언어 모델에서 차단
    pos       = np.arange(seq_len)
    distances = pos[:, None] - pos[None, :]            # shape: (seq_len, seq_len)

    # ── Step 3: 헤드별 편향 행렬 계산 ──────────────────────────
    biases = []
    for m in slopes:
        # 과거/현재: -m * |i - j|   (패널티, 거리에 비례)
        bias = -m * np.abs(distances)
        # 미래 토큰: -1e9 (softmax 후 0이 됨 → 사실상 차단)
        bias = np.where(distances < 0, -1e9, bias)
        biases.append(bias)

    return np.stack(biases)   # shape: (num_heads, seq_len, seq_len)

# ── 실행 및 확인 ────────────────────────────────────────────────
print("=" * 60)
print("ALiBi 편향 행렬 생성 및 확인")
print("=" * 60)

seq_len   = 6
num_heads = 4
alibi = alibi_bias(seq_len, num_heads)
slopes = 2.0 ** (-8.0 * np.arange(1, num_heads + 1) / num_heads)

print(f"\nshape: {alibi.shape}")
print(f"  = (헤드수={num_heads}, seq_len={seq_len}, seq_len={seq_len})")
print()

for h in range(num_heads):
    strength = "강한 패널티" if h == 0 else "약한 패널티" if h == num_heads-1 else "중간"
    print(f"헤드 {h}  (slope m={slopes[h]:.4f}, {strength}):")
    disp = alibi[h].copy()
    disp[disp < -100] = -999.0   # 읽기 쉽게 -999로 대체
    print(np.round(disp, 3))
    print()

print("읽는 법:")
print("  행[i] = Query 위치 i")
print("  열[j] = Key 위치 j")
print("  값    = Attention Score에 더할 편향 (음수=패널티)")
print()
print("  대각선 (i=j, 자기 자신): 0.0  (패널티 없음)")
print("  거리가 멀수록: 더 큰 음수 = 더 강한 패널티")
print("  -999 (실제는 -1e9): 미래 위치 → Softmax 후 0이 됨")

In [ ]:
# ============================================================
# 4-B. ALiBi 시각화 및 실제 어텐션 적용 예시
# ============================================================

# ── 시각화 ──────────────────────────────────────────────────────
seq_len_v = 10
n_heads_v = 4
alibi_v   = alibi_bias(seq_len_v, n_heads_v)
slopes_v  = 2.0 ** (-8.0 * np.arange(1, n_heads_v + 1) / n_heads_v)

fig, axes = plt.subplots(1, n_heads_v, figsize=(16, 4))
fig.suptitle("ALiBi Bias Matrix per Head\n"
             "(Green=no penalty | Red=strong penalty | White=future/masked)",
             fontsize=11, fontweight='bold')

for h, ax in enumerate(axes):
    bias_h = alibi_v[h].copy()
    bias_h[bias_h < -100] = np.nan    # -inf → NaN (흰색으로 표시)

    im = ax.imshow(bias_h, cmap='RdYlGn', aspect='auto', vmin=-1.2, vmax=0)
    ax.set_xlabel("Key pos (j)")
    ax.set_ylabel("Query pos (i)")
    strength = "Strong" if h == 0 else "Weak" if h == n_heads_v-1 else "Medium"
    ax.set_title(f"Head {h}\nm={slopes_v[h]:.4f} ({strength})")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks(range(seq_len_v))
    ax.set_yticks(range(seq_len_v))
    # 대각선(자기 자신) 표시
    for i in range(seq_len_v):
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1,
                                   fill=False, edgecolor='navy', linewidth=1.5))

plt.tight_layout()
plt.show()

print("시각화 해석:")
print("  녹색(0에 가까운 값): 패널티 없음 (가까운 토큰)")
print("  빨간색(큰 음수):    강한 패널티 (먼 토큰)")
print("  흰색: 미래 토큰 마스킹 (-1e9 → softmax 후 0)")
print("  파란 테두리: 자기 자신 (거리=0, 패널티 없음)")
print()
print("  Head 0 (slope 큼): 급격한 페널티 → 바로 옆 토큰 위주로 어텐션")
print("  Head 3 (slope 작음): 완만한 페널티 → 멀리도 어텐션 가능")

# ── 실제 어텐션 계산에 ALiBi 적용 ────────────────────────────────
print()
print("=" * 60)
print("실제 어텐션 계산에 ALiBi 적용 예시")
print("=" * 60)

np.random.seed(42)
seq_ex = 5
d_k_ex = 8
Q_ex   = np.random.randn(seq_ex, d_k_ex)
K_ex   = np.random.randn(seq_ex, d_k_ex)

# 기존 Attention Score
attn_score = Q_ex @ K_ex.T / np.sqrt(d_k_ex)  # shape: (5, 5)
print("\n① 기존 Attention Score (Q@K^T / sqrt(d_k)):")
print(np.round(attn_score, 3))

# ALiBi 편향 (헤드 0 사용)
alibi_ex = alibi_bias(seq_ex, n_heads_v)
bias_ex  = alibi_ex[0]   # 헤드 0의 편향 행렬

print("\n② ALiBi 편향 행렬 (헤드 0, m={:.4f}):".format(slopes_v[0]))
disp_ex = bias_ex.copy()
disp_ex[disp_ex < -100] = float('-inf')
print(np.round(disp_ex, 3))

# 최종 Score = 기존 + 편향
final_ex = attn_score + bias_ex
final_ex[final_ex < -100] = float('-inf')
print("\n③ 최종 Score (기존 + ALiBi 편향):")
print(np.round(final_ex, 3))

print()
print("→ 미래 위치(-inf): Softmax 후 0이 됩니다.")
print("  과거/현재 위치: 거리에 따라 차등 패널티 적용")
print("  가까운 토큰 (대각선 근처)이 자연스럽게 더 큰 어텐션을 받습니다.")

In [ ]:
# ============================================================
# 5. 위치 임베딩 방법론 총 비교 정리
# ============================================================

print("=" * 70)
print("위치 임베딩 방법론 총 비교")
print("=" * 70)

# 비교 테이블
methods = {
    "Sinusoidal": {
        "적용 위치":    "입력 임베딩에 더함(+)",
        "파라미터":     "없음 (0개)",
        "외삽 가능":    "가능",
        "상대위치 인코딩": "간접적 (내적)",
        "구현 난이도":  "쉬움",
        "사용 모델":    "원조 Transformer, BERT 일부",
    },
    "Learned": {
        "적용 위치":    "입력 임베딩에 더함(+)",
        "파라미터":     "O(max_len × d)개",
        "외삽 가능":    "불가 (IndexError)",
        "상대위치 인코딩": "아니오",
        "구현 난이도":  "매우 쉬움",
        "사용 모델":    "GPT-2, BERT",
    },
    "RoPE": {
        "적용 위치":    "Q, K 벡터 회전",
        "파라미터":     "없음 (0개)",
        "외삽 가능":    "가능 (변형 시 우수)",
        "상대위치 인코딩": "예 (수학적 보장)",
        "구현 난이도":  "중간",
        "사용 모델":    "LLaMA, Mistral, Falcon",
    },
    "ALiBi": {
        "적용 위치":    "Attention Score에 편향",
        "파라미터":     "없음 (0개)",
        "외삽 가능":    "매우 우수",
        "상대위치 인코딩": "예 (선형 페널티)",
        "구현 난이도":  "쉬움",
        "사용 모델":    "BLOOM, MPT",
    },
}

# 표 출력
keys = ["적용 위치", "파라미터", "외삽 가능", "상대위치 인코딩", "구현 난이도", "사용 모델"]
method_names = list(methods.keys())
col_w  = 24
row_lw = 16

header = f"{'항목':<{row_lw}}" + "".join(f"{m:^{col_w}}" for m in method_names)
sep    = "─" * (row_lw + col_w * len(method_names))
print()
print(header)
print(sep)
for key in keys:
    row = f"{key:<{row_lw}}"
    for m in method_names:
        val = methods[m][key]
        row += f"{val:^{col_w}}"
    print(row)
print(sep)

print()
print("=" * 70)
print("어떤 방법을 선택해야 할까?")
print("=" * 70)
print()
print("  ◆ 개념 학습 / 간단한 구현 실험")
print("      → Sinusoidal (수식이 명확하고 파라미터 없음)")
print()
print("  ◆ BERT/GPT 스타일 인코더/디코더 (고정 길이 입력)")
print("      → Learned (성능 좋고 구현 단순, 길이 제한 허용 시)")
print()
print("  ◆ 최신 LLM 구축 (LLaMA, Mistral 스타일)")
print("      → RoPE (상대 위치 수학적 보장 + 외삽 가능)")
print()
print("  ◆ 매우 긴 문서 / 외삽이 핵심인 경우")
print("      → ALiBi (외삽 성능 최우수 + 구현 가장 단순)")

print()
print("=" * 70)
print("핵심 정리 (5줄 요약)")
print("=" * 70)
print()
print("  1. 위치 임베딩은 Attention의 '순열 불변성' 문제를 해결합니다.")
print("  2. Sinusoidal / Learned: 토큰 임베딩에 위치 벡터를 더합니다 (절대 위치).")
print("  3. RoPE: Q/K를 위치에 따라 회전 → 상대 위치가 수학적으로 인코딩됩니다.")
print("  4. ALiBi: Attention Score에 거리 패널티 추가 → 가장 단순하고 외삽 우수.")
print("  5. 현재 주류 LLM (LLaMA, Mistral, Qwen 등)은 대부분 RoPE를 사용합니다.")

In [ ]:
# ============================================================
# 마지막: 4가지 방법의 핵심 특성 시각적 비교
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Positional Embedding Methods — Visual Comparison",
             fontsize=13, fontweight='bold')

methods_list = ["Sinusoidal", "Learned", "RoPE", "ALiBi"]

# ── 그래프 1: 특성 레이더 차트 (막대 형태로) ─────────────────────
ax = axes[0]
# 5개 특성에 대한 점수 (0~5)
features = ["외삽 능력", "상대 위치\n인코딩", "파라미터\n효율성", "구현\n단순성", "현재\n채택률"]
scores = {
    "Sinusoidal": [3, 2, 5, 4, 2],
    "Learned":    [1, 1, 2, 5, 3],
    "RoPE":       [4, 5, 5, 3, 5],
    "ALiBi":      [5, 4, 5, 5, 2],
}
colors = ['#3498db', '#e74c3c', '#27ae60', '#f39c12']
x      = np.arange(len(features))
width  = 0.2

for i, (method, color) in enumerate(zip(methods_list, colors)):
    offset = (i - 1.5) * width
    bars   = ax.bar(x + offset, scores[method], width=width,
                    label=method, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(features, fontsize=9)
ax.set_ylabel("Score (0-5)")
ax.set_title("Feature Comparison\n(Higher = Better)")
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 6)
ax.grid(alpha=0.25, axis='y')

# ── 그래프 2: 위치 유사도 패턴 (4가지 방법 비교) ─────────────────
ax = axes[1]
ref = 5
max_comp = 30

# Sinusoidal: 코사인 유사도
pe_comp = sinusoidal_encoding(max_comp, 32)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
sin_sims = [cosine_sim(pe_comp[ref], pe_comp[p]) for p in range(max_comp)]

# Learned: 학습 전 (랜덤) → 패턴 없음 (점선으로 표시)
np.random.seed(0)
lea_table = np.random.randn(max_comp, 32) * 0.02
lea_sims  = [cosine_sim(lea_table[ref], lea_table[p]) for p in range(max_comp)]

# ALiBi: 선형 감소 패턴 (slope=0.5)
m_ali     = 0.5
ali_sims  = [max(-1, 1 - m_ali * abs(ref - p)) for p in range(max_comp)]

# RoPE: 내적값 (정규화하여 비교)
np.random.seed(42)
q_vis = np.random.randn(32)
k_vis = np.random.randn(32)
q_rot_ref = apply_rope(q_vis, ref)
rope_dots = [np.dot(q_rot_ref, apply_rope(k_vis, p)) for p in range(max_comp)]
rope_norm = [d / (np.linalg.norm(q_vis) * np.linalg.norm(k_vis) + 1e-8)
             for d in rope_dots]

positions = range(max_comp)
ax.plot(positions, sin_sims,  label='Sinusoidal (cos sim)', color='#3498db', linewidth=1.8)
ax.plot(positions, lea_sims,  label='Learned (random init)', color='#e74c3c',
        linewidth=1.5, linestyle='--', alpha=0.7)
ax.plot(positions, rope_norm, label='RoPE (normalized dot)', color='#27ae60', linewidth=1.8)
ax.plot(positions, ali_sims,  label='ALiBi (linear decay)',  color='#f39c12',
        linewidth=1.8, linestyle='-.')

ax.axvline(x=ref, color='gray', linewidth=1.5, linestyle=':', label=f'Reference pos={ref}')
ax.set_xlabel("Token Position")
ax.set_ylabel("Similarity / Score")
ax.set_title(f"Position Similarity Pattern (ref=pos {ref})\n"
             "How each method encodes distance")
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.25)
ax.set_ylim(-0.5, 1.5)

plt.tight_layout()
plt.show()

print("그래프 2 해석:")
print("  Sinusoidal (파란색): 가까울수록 유사도 높음, 감소 패턴")
print("  Learned    (빨간점선): 학습 전에는 랜덤 (학습 후엔 패턴 형성)")
print("  RoPE       (초록색): 상대 거리에 따라 내적값 변화")
print("  ALiBi      (주황점선): 정확한 선형 감소 (거리에 정비례)")